# Validate the CSU compensation findings

This notebook checks the saved 2024 hierarchical lognormal fit. Run cells in order using the project `.venv` kernel. It reuses `csu_model.py`; it does not replace the full-data fit or change the model.

The checks answer different questions:
1. Do fitted group statistics resemble observed pay?
2. Can the model reproduce observed pay distributions?
3. How uncertain is a chosen bucket comparison?
4. How well does the same model predict records excluded from training?

All estimates concern annual pay excluding benefits and the sample's composition. They are not population-weighted or causal. No salary schedules are used. Tables are used instead of plots.


In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import arviz as az
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

MODEL_DIR = next(folder / relative
                 for folder in (Path.cwd(), *Path.cwd().parents)
                 for relative in ('Faculty/Modeling', '.')
                 if (folder / relative / 'csu_model.py').is_file())
sys.path.insert(0, str(MODEL_DIR))
from csu_model import DEFAULT_DATA, load_data
from csu_validation import (SAMPLING, predictive_draws, predictive_checks,
                            observed_comparisons, bucket_contrasts,
                            split_records, heldout_metrics)

FIT_DIR = MODEL_DIR / 'results'
OUTPUT = MODEL_DIR / 'results-validation'
OUTPUT.mkdir(exist_ok=True)
SEED = 4693
PPC_DRAWS = 500
LOW_PAY_THRESHOLD = 20_000  # Descriptive threshold only; records are retained.
BUCKET_A = 'Engineering'
BUCKET_B = 'Humanities & Languages'
COMPARISON_UNIVERSITY = 'CSU-wide'  # Or an exact campus name from coords['university'].
REUSE_HOLDOUT = True  # False refits the training split; never overwrites results/.


WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


## 1. Load the saved fit and verify its data

Stop if the source data differ from the file used for fitting. Reuse the original loader's checks, 2024 restriction, exclusion of `Other`, and category definitions. The saved convergence diagnostics concern sampling, not model adequacy.


In [2]:
metadata = json.loads((FIT_DIR / 'run_metadata.json').read_text(encoding='utf-8'))
source_hash = hashlib.sha256(DEFAULT_DATA.read_bytes()).hexdigest()
if source_hash != metadata['input_sha256']:
    raise ValueError('Input data changed since fitting. Refit the full-data model first.')
data, coords, audit = load_data()
idata = az.from_netcdf(FIT_DIR / 'posterior.nc')
aggregate_draws = xr.load_dataset(FIT_DIR / 'aggregate_draws.nc', engine='h5netcdf')
for dim in ('university', 'bucket', 'job'):
    if list(idata.posterior.coords[dim].values) != coords[dim]:
        raise ValueError(f'Category order differs for {dim}.')
estimates = pd.concat([pd.read_csv(FIT_DIR / 'university_bucket_estimates.csv'),
                       pd.read_csv(FIT_DIR / 'csu_bucket_estimates.csv')], ignore_index=True)
display(pd.DataFrame([metadata['convergence']]))
if not metadata['convergence']['passed']:
    print('WARNING: full-data fit failed convergence criteria; interpret results cautiously.')


{
  "input_rows": 103559,
  "year": 2024,
  "year_rows": 11060,
  "zero_pay": 0,
  "negative_pay": 0,
  "missing_or_nonnumeric_pay": 0,
  "infinite_pay": 0,
  "excluded_other": 500
}
{
  "input_rows": 103559,
  "year": 2024,
  "year_rows": 11060,
  "zero_pay": 0,
  "negative_pay": 0,
  "missing_or_nonnumeric_pay": 0,
  "infinite_pay": 0,
  "excluded_other": 500,
  "fitting_rows": 10560,
  "universities": 22,
  "buckets": 10,
  "jobs": 12,
  "repeated_name_campus_year_keys": 266,
  "empty_cells": 19
}


,divergences,max_rhat,min_bulk_ess,min_tail_ess,passed
0,0,1.002545,2644.723444,4572.050372,True


## 2. Compare fitted and observed group statistics

`observed` is calculated directly from the records; variance uses `ddof=0`. `estimate` is the posterior mean of the corresponding model-distribution statistic. `difference` is fitted minus observed, and `relative_difference` is that difference divided by observed.

These are in-sample descriptive comparisons. The credible bounds describe uncertainty about model statistics; they do not test whether the observed statistic must fall inside them. Empty cells have no observed statistic and remain explicitly marked as extrapolations.


In [3]:
comparisons = observed_comparisons(data, estimates)
comparisons.to_csv(OUTPUT / 'observed_vs_fitted.csv', index=False)
display(comparisons.loc[comparisons.university.eq('CSU-wide'),
                       ['bucket', 'statistic', 'n_records', 'observed', 'estimate',
                        'difference', 'relative_difference']].round(3))
print('Empty campus-bucket cells:', int(estimates.extrapolated.sum()))


Empty campus-bucket cells: 19


,bucket,statistic,n_records,observed,estimate,difference,relative_difference
660,"Agriculture, Food & Animal",mean,344,8.313060e+04,9.890116e+04,1.577056e+04,0.190
661,"Agriculture, Food & Animal",median,344,8.605800e+04,6.298794e+04,-2.307006e+04,-0.268
662,"Agriculture, Food & Animal",variance,344,2.115557e+09,1.275393e+10,1.063837e+10,5.029
663,Architecture & Environmental Design,mean,296,7.898748e+04,9.230387e+04,1.331639e+04,0.169
664,Architecture & Environmental Design,median,296,8.619100e+04,6.008197e+04,-2.610903e+04,-0.303
665,Architecture & Environmental Design,variance,296,2.252199e+09,1.059679e+10,8.344595e+09,3.705
666,"Business, Economics & Finance",mean,1165,1.291995e+05,1.495984e+05,2.039888e+04,0.158
667,"Business, Economics & Finance",median,1165,1.441940e+05,1.044478e+05,-3.974620e+04,-0.276
668,"Business, Economics & Finance",variance,1165,3.914988e+09,2.337093e+10,1.945594e+10,4.970
669,Computer Science,mean,565,1.109556e+05,1.329364e+05,2.198080e+04,0.198


## 3. Posterior predictive checks

For 500 joint posterior draws, generate one new pay value for every observed record using its actual job, campus, and bucket. NumPy's lognormal generator implements the same conditional distribution as the PyMC model, with both posterior parameter uncertainty and individual pay variability.

Compare the observed mean, median, variance, 5th/95th/99th percentiles, and fraction below $20,000 against the same statistics in the replicated datasets. The threshold only defines a diagnostic; it does not filter pay.

`replicated_lower_95` and `replicated_upper_95` are posterior predictive intervals for finite-sample statistics. An `observed_outside_95` flag identifies a discrepancy to investigate, not a classical significance test. Checks share data and are not independent; do not count flags as independent hypothesis rejections. Small groups are particularly noisy.


In [4]:
replicated = predictive_draws(idata.posterior, data, n_draws=PPC_DRAWS, seed=SEED)
ppc = predictive_checks(data, replicated, low_pay_threshold=LOW_PAY_THRESHOLD)
ppc.to_csv(OUTPUT / 'posterior_predictive_checks.csv', index=False)
statewide_ppc = ppc.loc[ppc.university.eq('CSU-wide')]
display(statewide_ppc.loc[statewide_ppc.statistic.isin(['mean', 'median', 'variance']),
                         ['bucket', 'statistic', 'observed', 'replicated_mean',
                          'replicated_lower_95', 'replicated_upper_95', 'observed_outside_95']].round(2))
display(statewide_ppc.groupby('statistic').agg(
    buckets_checked=('bucket', 'size'), outside_95=('observed_outside_95', 'sum')))
del replicated


,bucket,statistic,observed,replicated_mean,replicated_lower_95,replicated_upper_95,observed_outside_95
0,"Agriculture, Food & Animal",mean,8.313060e+04,9.883678e+04,8.674977e+04,1.107541e+05,True
1,"Agriculture, Food & Animal",median,8.605800e+04,6.307073e+04,5.425832e+04,7.171018e+04,True
2,"Agriculture, Food & Animal",variance,2.115557e+09,1.268391e+10,7.311942e+09,2.138680e+10,True
7,Architecture & Environmental Design,mean,7.898748e+04,9.227842e+04,7.984331e+04,1.066012e+05,True
8,Architecture & Environmental Design,median,8.619100e+04,6.008320e+04,5.118356e+04,6.957192e+04,True
9,Architecture & Environmental Design,variance,2.252199e+09,1.066900e+10,6.066823e+09,1.934105e+10,True
14,"Business, Economics & Finance",mean,1.291995e+05,1.496073e+05,1.409725e+05,1.598903e+05,True
15,"Business, Economics & Finance",median,1.441940e+05,1.045205e+05,9.757684e+04,1.124443e+05,True
16,"Business, Economics & Finance",variance,3.914988e+09,2.318619e+10,1.782075e+10,3.089232e+10,True
21,Computer Science,mean,1.109556e+05,1.327974e+05,1.202632e+05,1.472060e+05,True


,buckets_checked,outside_95
statistic,,
low_pay_fraction,10,1
mean,10,10
median,10,10
q05,10,6
q95,10,10
q99,10,10
variance,10,10


## 4. Evaluate a specific bucket comparison

The configurable comparison is **Engineering minus Humanities & Languages** across the sampled CSU records. Subtract the two group statistics within each matching posterior draw. This retains their posterior correlation.

A positive difference favors bucket A. `probability_a_greater_b` is the posterior probability that A exceeds B under this model. The interval is the difference's own 95% credible interval, not a comparison of overlapping group intervals. Variance differences have units of dollars squared; mean/median differences use dollars.

These contrasts preserve each group's observed campus and job composition. They are not adjusted discipline effects or causal effects. If you change the campus to one with an empty cell, the result involves extrapolation.


In [5]:
contrasts = bucket_contrasts(aggregate_draws, BUCKET_A, BUCKET_B, COMPARISON_UNIVERSITY)
contrasts.to_csv(OUTPUT / 'bucket_contrasts.csv', index=False)
display(contrasts.round(4))
display(estimates.loc[estimates.university.eq(COMPARISON_UNIVERSITY)
                      & estimates.bucket.isin([BUCKET_A, BUCKET_B]),
                      ['university', 'bucket', 'n_records', 'weight_source', 'extrapolated']])


,university,bucket_a,bucket_b,statistic,posterior_mean_difference,lower_95,upper_95,probability_a_greater_b
0,CSU-wide,Engineering,Humanities & Languages,mean,2.950431e+04,2.079625e+04,3.857756e+04,1.0
1,CSU-wide,Engineering,Humanities & Languages,median,2.654652e+04,2.036339e+04,3.301888e+04,1.0
2,CSU-wide,Engineering,Humanities & Languages,variance,4.588318e+09,2.281881e+09,7.191639e+09,1.0


,university,bucket,n_records,weight_source,extrapolated
225,CSU-wide,Engineering,502,observed_joint_university_job,False
226,CSU-wide,Humanities & Languages,2296,observed_joint_university_job,False


## 5. Fit on training records only

Hold out approximately 20% of normalized-name groups. All records sharing a stripped, case-folded name stay together globally, including across campuses. Names are a conservative identity proxy: different people can share a name, and spelling variants can still refer to one person.

Every job, campus, and bucket remains represented in training. If a random split would remove an entire category, one whole name group is returned to training; the split metadata reports these cases. Consequently this checks prediction within supported categories, not prediction for unseen jobs or campuses.

The held-out outcomes never enter the fit. The model structure and priors are reused unchanged, with four chains and 2,000 tuning plus 2,000 retained iterations per chain. The first run takes a few minutes. Subsequent runs reuse the separate training-only posterior if its data hash, split, and sampling settings match.


In [6]:
train, test, split_assignment, split_info = split_records(data)
holdout_path = OUTPUT / 'holdout_posterior.nc'
holdout_meta_path = OUTPUT / 'holdout_metadata.json'
if not REUSE_HOLDOUT or not (holdout_path.exists() and holdout_meta_path.exists()):
    subprocess.run([sys.executable, str(MODEL_DIR / 'csu_validation.py'),
                    '--output', str(OUTPUT)], check=True)
holdout_meta = json.loads(holdout_meta_path.read_text(encoding='utf-8'))
if (holdout_meta['input_sha256'] != source_hash
        or holdout_meta['sampling'] != SAMPLING
        or holdout_meta['split'] != split_info):
    raise ValueError('Cached holdout fit is stale. Set REUSE_HOLDOUT=False and rerun.')
saved_split = pd.read_csv(OUTPUT / 'holdout_split.csv')
pd.testing.assert_frame_equal(saved_split, split_assignment.reset_index(drop=True), check_dtype=False)
holdout = az.from_netcdf(holdout_path)
display(pd.DataFrame([split_info]).drop(columns=['categories_pinned_to_training']))
print('Categories requiring a whole name group to remain in training:',
      split_info['categories_pinned_to_training'])
display(pd.DataFrame([holdout_meta['convergence']]))
if not holdout_meta['convergence']['passed']:
    print('WARNING: training-only fit failed convergence criteria; held-out metrics are provisional.')


Categories requiring a whole name group to remain in training: [{'column': 'job', 'level': 'instructional faculty, summer session - state support'}]


,seed,requested_test_fraction,actual_test_fraction,train_records,test_records,train_name_groups,test_name_groups,grouping
0,4694,0.2,0.200947,8438,2122,8165,2040,Globally normalized name; proxy identity. No n...


,divergences,max_rhat,min_bulk_ess,min_tail_ess,passed
0,0,1.002968,1647.66927,2881.089203,True


## 6. Evaluate held-out predictions

For each unseen record, evaluate the posterior predictive mixture across 2,000 selected training-posterior draws. Numerically invert its CDF to obtain the predictive median and 50%/95% intervals. These intervals describe an individual pay record, not uncertainty about a group's mean.

- **MAE of predictive medians (dollars):** average absolute prediction error; smaller is better.
- **Coverage 50 / 95:** observed fraction of held-out pay values inside the corresponding intervals; compare with 0.50 / 0.95. Finite test samples introduce uncertainty, and these observations are not all independent people.
- **Mean 95% interval width (dollars):** shows how broad intervals must be to get that coverage. Broad intervals alone do not establish useful predictions.
- **Mean log predictive density:** a proper score using the full predictive density in dollars; higher is better when comparing the same records and units. There is no universal good cutoff.

This is a single split, not cross-validation, and does not establish population representativeness. Avoid repeatedly changing the model in response to this test set; reserve a fresh evaluation set for future revisions.


In [7]:
holdout_scores, holdout_record_predictions = heldout_metrics(
    holdout.posterior, test, n_draws=2000, seed=SEED+2)
holdout_scores['convergence_passed'] = holdout_meta['convergence']['passed']
holdout_scores.to_csv(OUTPUT / 'heldout_metrics.csv', index=False)
holdout_record_predictions.to_csv(OUTPUT / 'heldout_record_predictions.csv', index=False)
display(holdout_scores.round(3))


,group,n_records,mae_predictive_median_dollars,mean_log_predictive_density,coverage_50,coverage_95,mean_95_interval_width,convergence_passed
0,Overall,2122,29377.812,-12.416,0.664,0.957,338712.478,True
1,"Agriculture, Food & Animal",57,20443.453,-11.928,0.649,0.930,311722.030,True
2,Architecture & Environmental Design,73,25326.762,-12.134,0.712,0.959,297375.174,True
3,"Business, Economics & Finance",231,36998.196,-12.462,0.723,0.978,436190.505,True
4,Computer Science,122,35085.681,-12.484,0.705,0.984,411792.545,True
5,Education & Public Administration,200,29013.529,-12.459,0.555,0.890,256254.583,True
6,Engineering,107,28919.610,-12.287,0.710,0.944,422093.129,True
7,Humanities & Languages,442,26216.746,-12.360,0.640,0.950,309677.058,True
8,Medicine & Health Sciences,169,29504.054,-12.444,0.586,0.959,268804.150,True
9,Science & Mathematics,309,27699.377,-12.600,0.715,0.974,356770.961,True


## 7. Read the evidence together

Good R-hat/ESS and zero divergences support the sampling computation. Observed-versus-fitted comparisons and posterior predictive checks assess distributional fit. The training-only holdout evaluates predictions beyond fitting records. Bucket contrasts quantify a particular comparison only under the fitted assumptions.

A precise contrast can coexist with poor predictive fit. If checks show large discrepancies, describe the findings as conditional on a model that does not reproduce those features; do not treat convergence as evidence of substantive accuracy. No automatic model changes are made here.

Validation tables and the separate training posterior are saved in `results-validation/`. The original `results/` files remain unchanged.


In [8]:
validation_metadata = dict(input_sha256=source_hash, seed=SEED,
                           posterior_predictive_draws=PPC_DRAWS,
                           low_pay_threshold=LOW_PAY_THRESHOLD,
                           comparison=dict(university=COMPARISON_UNIVERSITY,
                                           bucket_a=BUCKET_A, bucket_b=BUCKET_B),
                           full_fit_convergence=metadata['convergence'],
                           holdout_convergence=holdout_meta['convergence'], split=split_info)
(OUTPUT / 'validation_metadata.json').write_text(json.dumps(validation_metadata, indent=2), encoding='utf-8')
print('Validation outputs:', OUTPUT)
display(pd.DataFrame({'file':sorted(p.name for p in OUTPUT.iterdir() if p.is_file())}))

# Evidence-based readout; diagnostic flags are descriptive rather than hypothesis tests.
mean_checks = statewide_ppc.loc[statewide_ppc.statistic.eq('mean')]
print(f"Statewide buckets with observed mean outside the predictive interval: {mean_checks.observed_outside_95.sum()}/{len(mean_checks)}")
if (mean_checks.replicated_mean > mean_checks.observed).all():
    print('The model overpredicts mean pay in every statewide bucket.')
if mean_checks.observed_outside_95.any():
    print('The current model misses observed distributional features. Treat its bucket comparisons as model-conditional findings, not validated compensation conclusions.')
print('Read held-out coverage together with interval widths and prediction error above.')


Validation outputs: C:\Users\wkmp2\4693\CSC4693\Faculty\Modeling\results-validation
Statewide buckets with observed mean outside the predictive interval: 10/10
The model overpredicts mean pay in every statewide bucket.
The current model misses observed distributional features. Treat its bucket comparisons as model-conditional findings, not validated compensation conclusions.
Read held-out coverage together with interval widths and prediction error above.


,file
0,bucket_contrasts.csv
1,heldout_metrics.csv
2,heldout_record_predictions.csv
3,holdout_metadata.json
4,holdout_parameter_diagnostics.csv
5,holdout_posterior.nc
6,holdout_split.csv
7,observed_vs_fitted.csv
8,posterior_predictive_checks.csv
9,validation_metadata.json
